In [ ]:
// #r "./binaries/BoSSSpad.dll"
// #r "./binaries/LsTest.dll"
#r "BoSSSpad.dll"
#r "LsTest.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Solution.LevelSetTools;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Application.LsTest;
Init();

# Job Creation and Deployment

Init Workflowmanagement and Project

In [ ]:
BoSSSshell.WorkflowMgm.Init("SwirlingFlow_SpatialConvergence");
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
var bpc = BoSSSshell.GetDefaultQueue();

Deploy Managed and Unmanaged Assemblies once

In [ ]:
using System.IO;

In [ ]:
DirectoryInfo NativeOverride = null;
DirectoryInfo ManagedOverride = null;
string RelManagedPath = null;

string mngdir = "SwirlingFlow_SpatialConvergence_" + DateTime.Now.ToString("MMMdd_HHmmss") + "_managed";
ManagedOverride = new DirectoryInfo(Path.Combine(bpc.DeploymentBaseDirectory, mngdir));
ManagedOverride.Create();
typeof(SolverWithLevelSetUpdaterTestCenter).Assembly.DeployAt(ManagedOverride);
RelManagedPath = "../" + mngdir + "/" + Path.GetFileName(typeof(SolverWithLevelSetUpdaterTestCenter).Assembly.Location);

string ntvdir = "SwirlingFlow_SpatialConvergence_" + DateTime.Now.ToString("MMMdd_HHmmss") + "_amd64";
NativeOverride = new DirectoryInfo(Path.Combine(bpc.DeploymentBaseDirectory, ntvdir));
NativeOverride.Create();
MetaJobMgrIO.CopyDirectoryRec(ilPSP.Environment.NativeLibraryDir, NativeOverride.FullName, null);

bpc.DeployRuntime = false; // we did this manually!

Create and Submit Jobs, we are loading the Controlfiles from Code

In [ ]:
int[] degrees = new int[] { 2, 3 };
int[] gridResolutions = new int[] { 1, 2, 3, 4 };
LevelSetEvolution[] lsEvos = new LevelSetEvolution[] { LevelSetEvolution.PrescribedVelocity, LevelSetEvolution.FastMarching, LevelSetEvolution.StokesExtension };

In [ ]:
foreach(var lsEvo in lsEvos) {
    foreach(int p in degrees) {
        foreach(int gridRes in gridResolutions) {
            // create job
            var C = BoSSS.Application.LsTest.LevelSetUnitTests.SwirlingFlowSpatialConvergence(p, gridRes, lsEvo, BoSSSshell.WorkflowMgm.CurrentProject, BoSSSshell.WorkflowMgm.DefaultDatabase.Path); // create as reference, just so settings as names etc. are really equal (avoid typos)
            Job j = new Job(C.SessionName, typeof(SolverWithLevelSetUpdaterTestCenter));// job name has to be equal to sessionname! for SessionInfoJobCorrelation to work!
            j.RetryCount = 2;
            j.MySetCommandLineArguments("--control", $"cs:BoSSS.Application.LsTest.LevelSetUnitTests.SwirlingFlowSpatialConvergence({p},{gridRes},BoSSS.Solution.LevelSetTools.LevelSetEvolution.{lsEvo.ToString()},\"{C.ProjectName}\", {"@\""+C.DbPath+"\""})"); // db path has to be valid at remote location as well! otherwise fix this somehow to pathatremote
            j.EnvironmentVars.Add(BoSSS.Foundation.IO.Utils.BOSSS_NATIVE_OVERRIDE, NativeOverride.FullName);
            j.EntryAssemblyRedirection = RelManagedPath;
            j.Activate(bpc); 
        }
    }
}

Wait for Jobs to finish

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(3600 * 24 * 3); // 3 days